In [1]:
!pip install optuna hyperopt -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 9.6 MB/s eta 0:00:00


In [3]:
from google.colab import files

uploaded = files.upload()

Saving bengaluru_house_prices.csv to bengaluru_house_prices.csv


In [4]:
import pandas as pd

df = pd.read_csv("bengaluru_house_prices.csv")

df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [5]:
df = pd.read_csv("bengaluru_house_prices.csv")
df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     13320 non-null  object 
 1   availability  13320 non-null  object 
 2   location      13319 non-null  object 
 3   size          13304 non-null  object 
 4   society       7818 non-null   object 
 5   total_sqft    13320 non-null  object 
 6   bath          13247 non-null  float64
 7   balcony       12711 non-null  float64
 8   price         13320 non-null  float64
dtypes: float64(3), object(6)
memory usage: 936.7+ KB


In [7]:
# نشوف القيم الناقصة في كل عمود
df.isnull().sum()

,0
area_type,0
availability,0
location,1
size,16
society,5502
total_sqft,0
bath,73
balcony,609
price,0


In [8]:
# عدد الصفوف والأعمدة
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

# أسماء الأعمدة
print(df.columns.tolist())

Rows: 13320
Columns: 9
['area_type', 'availability', 'location', 'size', 'society', 'total_sqft', 'bath', 'balcony', 'price']


In [9]:
# حذف الأعمدة اللي مش هنحتاجها
df = df.drop(['area_type', 'availability', 'society'], axis=1)

# حذف الصفوف اللي فيها قيم ناقصة
df = df.dropna()

# تحويل size إلى رقم (عدد غرف النوم)
df['bhk'] = df['size'].str.extract('(\d+)').astype(int)

# حذف size بعد استخراج BHK
df = df.drop('size', axis=1)

# تحويل location إلى أرقام باستخدام One-Hot Encoding
df = pd.get_dummies(df, columns=['location'], dtype=int)

# عرض شكل البيانات بعد التنظيف
print(df.shape)
df.head()

(12710, 1270)


<>:8: SyntaxWarning: invalid escape sequence '\d'
<>:8: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_675/3869561074.py:8: SyntaxWarning: invalid escape sequence '\d'
  df['bhk'] = df['size'].str.extract('(\d+)').astype(int)


,total_sqft,bath,balcony,price,bhk,location_ Anekal,location_ Banaswadi,location_ Basavangudi,location_ Bhoganhalli,location_ Devarabeesana Halli,...,"location_ravindra nagar, T.dasarahalli peenya",location_rr nagar,location_sankeswari,location_sapthagiri Layout,location_sarjapura main road,location_singapura paradise,location_t.c palya,location_tc.palya,location_vinayakanagar,location_whitefiled
0,1056,2.0,1.0,39.07,2,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2600,5.0,3.0,120.00,4,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1440,2.0,3.0,62.00,3,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1521,3.0,1.0,95.00,3,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1200,2.0,1.0,51.00,2,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [13]:
from sklearn.model_selection import train_test_split

# X = كل الأعمدة ماعدا السعر
X = df.drop('price', axis=1)

# y = السعر
y = df['price']

# تقسيم البيانات: 80% تدريب و20% اختبار
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (10168, 1269)
X_test: (2542, 1269)
y_train: (10168,)
y_test: (2542,)


In [14]:
import pandas as pd
import numpy as np

# نعيد تحميل الداتا من الملف من البداية
df = pd.read_csv("bengaluru_house_prices.csv")

# حذف الأعمدة غير المطلوبة
df = df.drop(['area_type', 'availability', 'society'], axis=1)

# تحويل total_sqft إلى رقم
def convert_sqft_to_num(x):
    try:
        if '-' in str(x):
            values = x.split('-')
            return (float(values[0]) + float(values[1])) / 2
        return float(x)
    except:
        return np.nan

df['total_sqft'] = df['total_sqft'].apply(convert_sqft_to_num)

# حذف الصفوف التي بها قيم ناقصة
df = df.dropna()

# استخراج عدد غرف النوم من size
df['bhk'] = df['size'].str.extract(r'(\d+)').astype(int)

# حذف size
df = df.drop('size', axis=1)

# تحويل location إلى أرقام
df = pd.get_dummies(df, columns=['location'], dtype=int)

print("Shape:", df.shape)
df.head()

Shape: (12668, 1264)


,total_sqft,bath,balcony,price,bhk,location_ Anekal,location_ Banaswadi,location_ Basavangudi,location_ Bhoganhalli,location_ Devarabeesana Halli,...,"location_ravindra nagar, T.dasarahalli peenya",location_rr nagar,location_sankeswari,location_sapthagiri Layout,location_sarjapura main road,location_singapura paradise,location_t.c palya,location_tc.palya,location_vinayakanagar,location_whitefiled
0,1056.0,2.0,1.0,39.07,2,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2600.0,5.0,3.0,120.00,4,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1440.0,2.0,3.0,62.00,3,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1521.0,3.0,1.0,95.00,3,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1200.0,2.0,1.0,51.00,2,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [15]:
from sklearn.model_selection import train_test_split

# Features
X = df.drop('price', axis=1)

# Target: سعر البيت
y = df['price']

# تقسيم البيانات إلى تدريب واختبار
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (10134, 1263)
Testing data: (2534, 1263)


In [16]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# إنشاء الموديل
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# تدريب الموديل
model.fit(X_train, y_train)

# Prediction
y_pred = model.predict(X_test)

# Evaluation
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("MSE:", mse)
print("R2 Score:", r2)

MAE: 29.621433203481907
MSE: 7994.143882238145
R2 Score: 0.6278779677771376


In [17]:
import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

def objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "max_depth": trial.suggest_int("max_depth", 5, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
        "max_features": trial.suggest_float("max_features", 0.5, 1.0),
    }

    model = RandomForestRegressor(
        **params,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    mse = mean_squared_error(y_test, predictions)

    return mse


study = optuna.create_study(direction="minimize")

study.optimize(objective, n_trials=30)

print("Best Parameters:")
print(study.best_params)

print("\nBest MSE:")
print(study.best_value)

[I 2026-08-14 23:07:53,936] A new study created in memory with name: no-name-98da6bf8-49a8-49cc-b52a-4da7f3c79f7c
[I 2026-08-14 23:08:21,418] Trial 0 finished with value: 7550.254867586855 and parameters: {'n_estimators': 183, 'max_depth': 13, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 0.9737626917073725}. Best is trial 0 with value: 7550.254867586855.
[I 2026-08-14 23:08:50,727] Trial 1 finished with value: 7261.988094472365 and parameters: {'n_estimators': 221, 'max_depth': 14, 'min_samples_split': 2, 'min_samples_leaf': 5, 'max_features': 0.8673787487618516}. Best is trial 1 with value: 7261.988094472365.
[I 2026-08-14 23:09:13,915] Trial 2 finished with value: 7017.207637585347 and parameters: {'n_estimators': 197, 'max_depth': 20, 'min_samples_split': 2, 'min_samples_leaf': 3, 'max_features': 0.5351787567607891}. Best is trial 2 with value: 7017.207637585347.
[I 2026-08-14 23:09:54,756] Trial 3 finished with value: 7478.988389370877 and parameters: {'n_estimat

Best Parameters:
{'n_estimators': 104, 'max_depth': 24, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 0.598906445234163}

Best MSE:
6865.073495090496


In [18]:
best_params = study.best_params

optuna_model = RandomForestRegressor(
    **best_params,
    random_state=42,
    n_jobs=-1
)

optuna_model.fit(X_train, y_train)

optuna_pred = optuna_model.predict(X_test)

optuna_mae = mean_absolute_error(y_test, optuna_pred)
optuna_mse = mean_squared_error(y_test, optuna_pred)
optuna_r2 = r2_score(y_test, optuna_pred)

print("Optuna Results")
print("MAE:", optuna_mae)
print("MSE:", optuna_mse)
print("R2 Score:", optuna_r2)

Optuna Results
MAE: 31.944713372527826
MSE: 6865.073495090496
R2 Score: 0.6804354364913993


In [19]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials

# مساحة البحث عن أفضل Hyperparameters
space = {
    'n_estimators': hp.quniform('n_estimators', 50, 300, 1),
    'max_depth': hp.quniform('max_depth', 5, 30, 1),
    'min_samples_split': hp.quniform('min_samples_split', 2, 10, 1),
    'min_samples_leaf': hp.quniform('min_samples_leaf', 1, 5, 1),
    'max_features': hp.uniform('max_features', 0.5, 1.0)
}

def objective_hyperopt(params):

    params['n_estimators'] = int(params['n_estimators'])
    params['max_depth'] = int(params['max_depth'])
    params['min_samples_split'] = int(params['min_samples_split'])
    params['min_samples_leaf'] = int(params['min_samples_leaf'])

    model = RandomForestRegressor(
        **params,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    mse = mean_squared_error(y_test, predictions)

    return {
        'loss': mse,
        'status': STATUS_OK
    }


trials = Trials()

best = fmin(
    fn=objective_hyperopt,
    space=space,
    algo=tpe.suggest,
    max_evals=30,
    trials=trials,
    rstate=np.random.default_rng(42)
)

print("Best Parameters from Hyperopt:")
print(best)

100%|██████████| 30/30 [12:43<00:00, 25.45s/trial, best loss: 6741.39365953954]
Best Parameters from Hyperopt:
{'max_depth': np.float64(15.0), 'max_features': np.float64(0.7315723121682772), 'min_samples_leaf': np.float64(2.0), 'min_samples_split': np.float64(10.0), 'n_estimators': np.float64(197.0)}


In [20]:
# تحويل Best Parameters إلى النوع الصحيح

hyperopt_params = {
    'max_depth': int(best['max_depth']),
    'max_features': best['max_features'],
    'min_samples_leaf': int(best['min_samples_leaf']),
    'min_samples_split': int(best['min_samples_split']),
    'n_estimators': int(best['n_estimators'])
}

# إنشاء الموديل بأفضل Parameters
hyperopt_model = RandomForestRegressor(
    **hyperopt_params,
    random_state=42,
    n_jobs=-1
)

# التدريب
hyperopt_model.fit(X_train, y_train)

# Prediction
hyperopt_pred = hyperopt_model.predict(X_test)

# Evaluation
hyperopt_mae = mean_absolute_error(y_test, hyperopt_pred)
hyperopt_mse = mean_squared_error(y_test, hyperopt_pred)
hyperopt_r2 = r2_score(y_test, hyperopt_pred)

print("Hyperopt Results")
print("MAE:", hyperopt_mae)
print("MSE:", hyperopt_mse)
print("R2 Score:", hyperopt_r2)

Hyperopt Results
MAE: 32.37847869334575
MSE: 6741.39365953954
R2 Score: 0.6861926498250834


In [21]:
print("Optuna Results")
print("MAE:", optuna_mae)
print("MSE:", optuna_mse)
print("R2 Score:", optuna_r2)

Optuna Results
MAE: 31.944713372527826
MSE: 6865.073495090496
R2 Score: 0.6804354364913993


In [22]:
comparison = pd.DataFrame({
    'Model': ['Baseline', 'Optuna', 'Hyperopt'],
    'MAE': [mae, optuna_mae, hyperopt_mae],
    'MSE': [mse, optuna_mse, hyperopt_mse],
    'R2 Score': [r2, optuna_r2, hyperopt_r2]
})

comparison

,Model,MAE,MSE,R2 Score
0,Baseline,29.621433,7994.143882,0.627878
1,Optuna,31.944713,6865.073495,0.680435
2,Hyperopt,32.378479,6741.393660,0.686193
